# Vibrational density of states (VDOS) evolution
Eigenvalue spectra of jammed states across training rounds.

In [ ]:
import h5py
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.cm as cm

H5_PATH = "campaign_analysis.h5"
plt.rcParams.update({'font.size': 10, 'axes.grid': True, 'grid.alpha': 0.3})

f = h5py.File(H5_PATH, 'r')
v = f['vdos']
rounds      = v['rounds'][:]
edges       = v['omega_edges'][:]   # (B+1,)
counts      = v['counts'][:]        # (S, B)
omega_star  = v['omega_star'][:]
n_states    = v['n_states'][:]
bin_centres = 0.5 * (edges[:-1] + edges[1:])

print(f"VDOS: {len(rounds)} rounds, ω range [{edges[0]:.3f}, {edges[-1]:.3f}]")

In [ ]:
# ---- waterfall heatmap: rounds × omega ----
fig, ax = plt.subplots(figsize=(10, 5))
im = ax.imshow(
    counts, aspect='auto', origin='lower',
    extent=[edges[0], edges[-1], rounds[0], rounds[-1]],
    cmap='viridis', interpolation='nearest'
)
plt.colorbar(im, ax=ax, label='normalised D(ω)')
ax.set_xlabel('ω (vibrational frequency)')
ax.set_ylabel('training round')
ax.set_title('VDOS evolution over training')
fig.tight_layout()
fig.savefig('vdos_waterfall.pdf', bbox_inches='tight')
plt.show()

In [ ]:
# ---- overlaid VDOS at 5 key rounds ----
n = len(rounds)
key_idx = [0, n//4, n//2, 3*n//4, n-1]
cmap = cm.get_cmap('plasma', len(key_idx))

fig, ax = plt.subplots(figsize=(8, 4))
for j, i in enumerate(key_idx):
    ax.plot(bin_centres, counts[i], lw=1.5, color=cmap(j), label=f'round {rounds[i]}')
ax.set_xlabel('ω'); ax.set_ylabel('normalised D(ω)')
ax.set_title('VDOS at key training stages')
ax.legend(fontsize=8)
fig.tight_layout()
fig.savefig('vdos_key_rounds.pdf', bbox_inches='tight')
plt.show()

In [ ]:
# ---- omega_star vs round with eval_dphi dual axis ----
fig, ax1 = plt.subplots(figsize=(9, 3))
ax1.plot(rounds, omega_star, 'o-', ms=3, lw=1, color='steelblue', label='ω*')
ax1.set_xlabel('round'); ax1.set_ylabel('ω* (5th pct eigenfreq)', color='steelblue')
ax1.tick_params(axis='y', labelcolor='steelblue')

if 'summary' in f and 'eval_dphi' in f['summary']:
    sr = f['summary']['round'][:]
    dphi = f['summary']['eval_dphi'][:]
    ax2 = ax1.twinx()
    ax2.plot(sr, dphi, '.', ms=2, alpha=0.4, color='tomato')
    ax2.axhline(0, color='tomato', lw=0.8, ls='--', alpha=0.6)
    ax2.set_ylabel('eval ⟨φ − φ_null⟩', color='tomato')
    ax2.tick_params(axis='y', labelcolor='tomato')

ax1.set_title('ω* and eval Δφ vs training round')
fig.tight_layout()
fig.savefig('vdos_omega_star.pdf', bbox_inches='tight')
plt.show()

In [ ]:
# ---- n_states per round (data quality) ----
fig, ax = plt.subplots(figsize=(9, 2.5))
ax.bar(rounds, n_states, width=max(1, rounds[1]-rounds[0]) if len(rounds) > 1 else 1, align='center')
ax.set_xlabel('round'); ax.set_ylabel('# jammed states')
ax.set_title('States per sampled round (data quality)')
fig.tight_layout()
plt.show()